In [ ]:
#=============================================== 一、查询产品及条款 ===============================================

from app.infra.database import AsyncSessionFactory
from app.modules.product.service import ProductService

async with AsyncSessionFactory() as session:
    service = ProductService(session)
    # 查询20条
    products= await service.list_products("critical_illness")

# 取一个测试
product = products[0]
print(product)

#=============================================== 二、文档解析 ===============================================

from mineru import MinerU
from app.core.config import settings

# 创建客户端，Precision 模式，需要Token
client = MinerU(settings.rag.mineru_token)
# 解析文件
result = client.extract(f"../../data/raw/kb/{product.clause_name}")

#=============================================== 三、文档优化 ===============================================

from langchain.chat_models import init_chat_model

MARKDOWN_OPTIMIZATION_PROMPT = """
下面这份Markdown文档是从保险条款PDF解析得来，由于PDF中的各个小节是以表格形式存在，所以解析时出现错乱。你分析内容，帮我转为格式正确的Markdown，特别是标题编号要正确。
- 标题等级要从1级标题开始，逐层增加，目录和文档名不计入标题等级。
- 输出结果中不要包含正文开始之前的部分。
- 输出结果只包含Markdown正文，不要解释或代码围栏。
- 不要修改保险条款。
""".strip()

llm = init_chat_model(
    "deepseek-flash",
    api_key=settings.llm.api_key,
    extra_body={'thinking': {'type': 'disabled'}},
    max_tokens=10000
)

response = llm.invoke([
    {"role": "system", "content": MARKDOWN_OPTIMIZATION_PROMPT},
    {"role": "user", "content": result.markdown},
])
markdown = response.content
print(markdown)
#=============================================== 四、文档切分 ===============================================

from uuid import uuid4
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from app.rag.models import ParentChunk

# 1.准备TextSpliter
# 1.1.markdown切分器，切父块
markdown_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=[
                ("#", "h1"),
                ("##", "h2"),
                ("###", "h3"),
                ("####", "h4"),
            ],
            # 文本片段中中是否携带标题信息,不要携带标题，因它不完整，我们后续给它补上完整的
            strip_headers=True,
        )
# 1.2.递归切分器，切子块
child_splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            separators=["\n\n", "\n", "。", "；", ";", "，", ","],
            # 让每个 separator切割符 都保留在切分后它所属文本块的末尾
            keep_separator="end",
        )

# 2.先用markdown切分，得到父块小节信息
sections = markdown_splitter.split_text(markdown)
# 3.处理父子块
parent_chunks: list[ParentChunk] = [] # 父块是要存入pg数据库的parent_chunk表，要使用orm模型
child_chunks: list[Document] = [] # 子块是要存入milvus的，数据类型为Document

# 3.1.遍历父块小节
for section in sections:
    # 3.1.生成章节路径，section.metadata就是每一级标题的字典，values刚好得到标题列表
    section_path = list(section.metadata.values())
    # 3.2.组织 parent_chunk
    parent = ParentChunk(
        # 不要让数据库生成，提前生成好，给后续子块的parent_id用。
        id=uuid4(),
        product_id=product.id,
        content=section.page_content,
        section_path=section_path,
        clause_name=product.clause_name
    )
    parent_chunks.append(parent)

    # 3.3.重新组织section的metadata，会作为所有子块的metadata
    section.metadata = {
        "product_id": product.id,
        "parent_id": str(parent.id)
    }
    # 3.4.切分section，生成child的Document列表，并且沿用section的metadata
    children = child_splitter.split_documents([section])
    # 3.5.把父的section_path拼接到子content中，使信息完整
    section_path_header = "\n".join(section_path) + "\n"
    for c in children:
        # 每一个子块中都要带上 章节路径
        c.page_content = section_path_header + c.page_content
    child_chunks.extend(children)

#=============================================== 五、存入向量库 ===============================================
from langchain_milvus import Milvus, BM25BuiltInFunction
from app.core.config import settings
# 【修改点】将 DashScopeEmbeddings 替换为 OllamaEmbeddings
# from langchain_community.embeddings import DashScopeEmbeddings
from langchain_ollama import OllamaEmbeddings
from dotenv import load_dotenv
load_dotenv()
# 1. 初始化本地的 Ollama 向量模型
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",
    dimensions=1024
)
# # 1.初始化向量模型
# embeddings = DashScopeEmbeddings(
#     model="text-embedding-v4"
# )

# 2、初始化 VectorStore，会自动创建Collection
vector_store = Milvus(
    embedding_function=embeddings,
    collection_name="insurance_collection",
    builtin_function=BM25BuiltInFunction(analyzer_params={"type": "chinese"}),
    vector_field=["dense", "sparse"],
    connection_args={"uri": settings.rag.milvus_url},
    auto_id=True,
)

# 3. 删除旧文档, 防止重复导入
vector_store.delete(expr=f"product_id=={product.id}")
# 4. 添加新文档
vector_store.add_documents(child_chunks)
print("✅ 数据存入 Milvus 成功！")

In [5]:
# 六、父块入数据库
from app.infra.database import AsyncSessionFactory
from app.rag.repository import ParentChunkRepository

async with AsyncSessionFactory() as session:
    async with session.begin():
        repository = ParentChunkRepository(session)
        # 先删除旧数据
        await repository.delete_by_product_id(product.id)
        # 再添加新数据
        repository.add_all(parent_chunks)

ImportError: cannot import name 'ParentChunkRepository' from 'app.rag.repository' (E:\RAG_Study\hm-insurance\agent-service\app\rag\repository.py)

In [6]:
# 六、父块入数据库（绕过缓存的黑科技版）
from app.infra.database import AsyncSessionFactory
from app.rag.models import ParentChunk # 直接导入模型
from sqlalchemy import delete

async with AsyncSessionFactory() as session:
    async with session.begin():
        # 1. 先删除旧数据（直接执行 SQL 删除）
        await session.execute(
            delete(ParentChunk).where(ParentChunk.product_id == product.id)
        )

        # 2. 把内存里的 parent_chunks 直接丢进 Session
        session.add_all(parent_chunks)

print("✅ 成功！43条父块已完美存入 PostgreSQL！")

✅ 成功！43条父块已完美存入 PostgreSQL！


In [21]:
import importlib
import app.rag.repository
from app.infra.database import AsyncSessionFactory
from app.rag.models import ParentChunk
from sqlalchemy import select
# 强制重新加载这个模块
importlib.reload(app.rag.repository)
from app.rag.repository import ParentChunkRepository

async with AsyncSessionFactory() as session:
    async with session.begin():
        repository = ParentChunkRepository(session)
        chunks = await repository.list_by_ids(["048da4bb-52e7-44dd-81dc-bc90f715f477"])

for c in chunks:
    print(c)

async with AsyncSessionFactory() as session:
    # 直接查前 5 条，不加任何 where 条件
    result = await session.scalars(select(ParentChunk).limit(100))
    chunks = result.all()

print(f"查到了 {len(chunks)} 条数据")
for c in chunks:
    print(f"ID: {c.id}, 产品: {c.clause_name}")

ParentChunk({'product_id': 2, 'clause_name': '复星联合优选K重疾险(达尔文12号)条款.pdf', 'content': '自投保人签收本合同或收到本合同电子保单之日起（二者较早之日），投保人有15日的犹豫期。在此期间，请投保人认真审阅本合同，如果投保人认为本合同与投保人的需求不相符并在犹豫期内未发生理赔，可以在此期间提出解除本合同，本公司将向投保人无息退还保险费。犹豫期内解除本合同时，投保人须填写解除合同申请书，并提供保险合同和有效身份证件<sup>5</sup>，自本公司收到投保人的解除合同申请书之日起，本合同即被解除，对于合同解除前发生的保险事故，本公司不承担保险责任。', 'id': UUID('048da4bb-52e7-44dd-81dc-bc90f715f477'), 'section_path': ['复星联合优选K重大疾病保险（互联网）条款', '1 合同订立', '1.4 犹豫期'], 'created_at': '2026-10-31 02:29:37'})
查到了 100 条数据
ID: eb01e61e-5d79-4f7d-828e-e00926cd7b4d, 产品: 信美相互星辰守护B款少儿重疾险条款.pdf
ID: 8e2747b2-cbfd-40c0-b02d-767285991c6b, 产品: 信美相互星辰守护B款少儿重疾险条款.pdf
ID: e091b666-d8ce-4d77-a4f0-39f558f1cdaf, 产品: 信美相互星辰守护B款少儿重疾险条款.pdf
ID: 549729c5-9765-469e-9f13-06aa5b5d2a08, 产品: 信美相互星辰守护B款少儿重疾险条款.pdf
ID: f87d22fb-1380-422e-9d77-519d107044db, 产品: 信美相互星辰守护B款少儿重疾险条款.pdf
ID: b3f9ac22-98aa-47c8-bf9d-293459169094, 产品: 信美相互星辰守护B款少儿重疾险条款.pdf
ID: 60760ab4-eb5e-44fb-b892-89a98d6dd5df, 产品: 信美相互星辰守护B款少儿重疾险条款.pdf
ID: 989b5e59-a90d-44ea-a567-126d3111e122, 

In [22]:
# 八、检索
from app.rag.retriever import ParentChunkRetriever

retriever = ParentChunkRetriever(vector_store)

parent_chunks = await retriever.retrieve(
    query="保险合同的等待期是多久？",
    k=5,
    product_id=product.id,
)

for parent_chunk in parent_chunks:
    print("="*80)
    print("product_id =", parent_chunk.product_id)
    print("clause_name =", parent_chunk.clause_name)
    print("section_path =", parent_chunk.section_path)
    print("content =", parent_chunk.content)

2026-10-31 11:00:41 [info     ] RAG Retriever 初始化完成✅️         
product_id = 1
clause_name = 信美相互星辰守护B款少儿重疾险条款.pdf
section_path = ['1. 我们保什么', '1.4 等待期']
content = 自本合同生效（或最后复效）之日起180日（含第180日）为等待期。被保险人在等待期内确诊非因意外伤害（见9.1）导致患有本合同所定义的重度疾病，我们不承担保险责任，但向您无息退还已交纳的本合同的保险费，本合同终止。被保险人在等待期内确诊非因意外伤害导致患有本合同所定义的中度疾病，我们不承担保险责任，中度疾病保险金、首次中度疾病意外保险金、中度疾病豁免保险费、首次中度疾病关爱保险金（若选择投保首次疾病关爱保险金）和首次中度疾病陪护保险金（若选择投保首次疾病陪护保险金）责任终止，本合同继续有效。被保险人在等待期内确诊非因意外伤害导致患有本合同所定义的轻度疾病，我们不承担保险责任，轻度疾病保险金、首次轻度疾病意外保险金、轻度疾病豁免保险费、首次轻度疾病关爱保险金（若选择投保首次疾病关爱保险金）和首次轻度疾病陪护保险金（若选择投保首次疾病陪护保险金）责任终止，本合同继续有效。被保险人在等待期内确诊患有本合同所定义的严重肥胖特定合并症（见9.2），我们不承担保险责任，严重肥胖手术关爱保险金责任终止，本合同继续有效。被保险人在等待期内确诊非因意外伤害导致患有本合同所定义的严重继发性脊柱侧弯（见9.3），我们不承担保险责任，脊柱侧弯矫正手术（见9.4）关爱保险金责任终止，本合同继续有效。被保险人在等待期内非因意外伤害导致身故或全残（见9.5）（若选择投保身故或全残保险金），我们不承担保险责任，但向您无息退还已交纳的本合同的保险费，本合同终止。  
被保险人因意外伤害导致保险事故的无等待期。  
本合同所定义的重度疾病、中度疾病和轻度疾病分别指载明于本合同"8.1 重度疾病"、"8.2 中度疾病"和"8.3 轻度疾病"中的疾病、疾病状态或手术。
product_id = 1
clause_name = 信美相互星辰守护B款少儿重疾险条款.pdf
section_path = ['1. 我们保什么', '1.3 保险期间']
content = 本合同的保险期间在